# Tutorial 160 — Create a custom QAChat tool: a research briefing

Most questions in QAChat are answered by retrieval plus the answering model.
Some requests need more than that: an exact count, every mention of a term,
or a structured report that must follow a fixed format. QAChat handles these
with **tools**: Python modules that perform one explicit operation for an
intent (tutorial 150 explains project types and intents).

This tutorial builds on the tool QAChat ships as its reference example,
`generate_paper_summary`. It turns one paper into a one-page summary that
follows a Markdown **template**, by default Nature's *Research Briefing*
(a short overview that unpacks a paper for a broad scientific audience). It
checks the structure in Python, fixes what misses a limit, and saves the
result in the project's `outputs/` folder.

Why a tool and not just a prompt? A tool is repeatable, has a defined input
and output, can be tested without a model, and can **check** its result
instead of trusting that the model followed the instructions. It can also
keep the model away from what it must not invent: here, the citation and
the references are copied from the paper by code.

The pieces, from configuration to code:

```
question → router → intent (project_types/literature.json)
                       ↓ names a tool
              tool registry (qachat/tools/registry.py)
                       ↓
     tool module (qachat/tools/generate_paper_summary.py)
          uses: outputs, take_notes, report templates (templates/*.md)
                       ↓
            answer + evidence + saved file in outputs/
```

## 1. Prepare QAChat

Each tutorial is independently runnable. These cells locate the repository,
make the local package importable, and call `prepare_notebook()` to verify the
project `.venv` environment. If Jupyter asks you to select the printed `.venv`
kernel, select it and rerun the cell.

In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")

In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)

In [ ]:
import json

import qachat
from IPython.display import Markdown, display

## 2. Where a tool is switched on: the project type

An intent in a project type (`qachat/project_types/literature.json`) can
name a **tool** and its settings. The router never sees Python: it only
chooses the intent, and the configuration decides what runs.

In [ ]:
from qachat.project_type import PROJECT_TYPES_DIR

literature = json.loads(
    (PROJECT_TYPES_DIR / "literature.json").read_text(encoding="utf-8")
)

Inspect the intent that selects the research briefing tool.

In [ ]:
print(json.dumps(literature["intents"]["paper_summary"], indent=2))

The registry is the controlled boundary between that configuration and code.
A project type can only name tools that are registered, and each tool
validates its own settings when the project type is loaded.

In [ ]:
from qachat.tools import TOOL_REGISTRY

for name, tool in TOOL_REGISTRY.items():
    print(f"{name:24} {tool.summary}")

## 3. The contract: what a tool receives and returns

Every tool has the same shape:

- a **settings** dataclass with `parse_settings(data, where)`;
- `run(settings, request, context) -> ToolResult`.

`ToolRequest` is what the user asked. `ToolContext` carries QAChat's
services as plain functions (indexed chunks, ranking, the models, progress,
confirmation, tracing), so a tool never imports the RAG session and can be
tested with simple stand-ins. `ToolResult` is the answer, the evidence it
cites, and a summary saved with the turn.

In [ ]:
from dataclasses import fields
from qachat.tools import ToolContext, ToolRequest, ToolResult

Inspect the request: the question and the earlier turns it refers to.

In [ ]:
print([field.name for field in fields(ToolRequest)])

Inspect the context: the backend services available to a tool.

In [ ]:
print([field.name for field in fields(ToolContext)])

Inspect the result: what a tool returns to QAChat.

In [ ]:
print([field.name for field in fields(ToolResult)])

## 4. Templates: a format anyone can edit

Report formats are Markdown files, not Python. Every project has a
`templates/` folder next to `sources/` and `outputs/` (never indexed), which
starts with a copy of the templates QAChat ships in
`qachat/tools/templates/`. Edit a template there, or add your own `.md`
file, and ask for it by name: *"Summarize smith2020.pdf with the
lab_summary template"*.

Here is the shipped research briefing:

In [ ]:
from qachat.tools.report_template import SHIPPED_TEMPLATES_DIR

briefing_file = SHIPPED_TEMPLATES_DIR / "research_briefing.md"
print(briefing_file.read_text(encoding="utf-8"))

A template follows a few rules, so QAChat can check a report against it:

1. **Front matter** (optional): `key: value` lines between two `---`
   lines at the top. Keys: `title`, `description`, `max_words` (all
   sections together), and `reference` (where the format comes from).
2. **Headline**: the `#` line. With a limit, `# Headline (max 90
   characters)`, the model writes the headline; without one, the line is
   the report's fixed title, such as `# Lab Summary`.
3. **Sections**: `## Name (limits)` lines, in the order the report uses.
   The text under a heading is the instruction for writing that section.
4. **Limits**, in brackets at the end of a heading: `(60–150 words)`,
   `(max 100 words)`, `(2–3 items)`, or `(3 items)`. A heading without
   brackets has no limit.
5. **Labelled lines**: `**Label:** {citation}` puts a one-line field in
   the report.
6. **Placeholders**: `{citation}` is the paper's citation, copied by code
   from the file's metadata and never written by the model.
   `{references}` in a section's text means its items are chosen from the
   paper's own reference list and copied word for word; such a section
   needs an item limit.

`load_template` reads a template into the structure the tool uses:

In [ ]:
from qachat.tools.report_template import load_template

template = load_template("research_briefing")

Inspect the headline and total word limits.

In [ ]:
print(f"{template.title}: headline of at most "
      f"{template.headline_characters} characters, at most "
      f"{template.max_total_words} words in total")

Inspect the section limits and their content sources.

In [ ]:
for section in template.sections:
    limit = section.words or section.items or ""
    print(f"- {section.name} {limit} ({section.source})")

Writing your own is the same. This one has a fixed title, two prose
sections, and a reading list taken from the paper's references:

In [ ]:
lab_summary = """---
title: Lab Summary
max_words: 300
---

# Lab Summary

## Question (max 40 words)
What the paper asks, in one or two sentences.

## Findings (60–120 words)
What it found, with the key numbers.

## Read Next (2 items)
Two papers to read next, from {references}.
"""

Parse the text into a checked template structure.

In [ ]:
from qachat.tools.report_template import parse_template

mine = parse_template(lab_summary, "lab_summary")

Inspect the prose sections and the reference-list section.

In [ ]:
for section in mine.sections:
    print(f"- {section.name}: words {section.words}, items "
          f"{section.items}, from {section.source}")

A template that breaks a rule is reported with the line, so it is quick to
fix:

In [ ]:
try:
    parse_template("## Findings (sixty words)\nWhat it found.", "draft")
except ValueError as error:
    print(error)

## 5. Checking the structure in Python

Asking a model to "follow this format exactly" is not a guarantee.
`check_report` verifies the result: every section present, not repeated,
not empty, in order, within its word or item limit, and a headline within
its length. A report built to fit passes.

Define a helper that supplies content within one section limit.
This is test data, not a real report.

In [ ]:
def filler(section):
    """Content that fits one section's limit."""

    if section.items:
        return "- First paper.\n- Second paper."
    low, high = section.words
    return " ".join(["word"] * ((low + high) // 2))

Assemble a sample report with the template headings.

In [ ]:
parts = ["# Plants reveal stress before it shows"]
for section in template.sections:
    if section.inline:
        parts.append(f"**{section.name}:** Smith, J. et al. A title.")
    else:
        parts.append(f"## {section.name}\n{filler(section)}")

Validate the assembled report in Python.

In [ ]:
from qachat.tools.report_template import check_report

fitting = "\n\n".join(parts)
print("valid:", check_report(fitting, template).valid)

An incomplete or overlong report fails, and the check says exactly what is
wrong. The tool uses these sentences to ask the model for a targeted fix.

In [ ]:
broken = "# " + "x" * 100 + "\n## The Reason\nToo short."
for problem in check_report(broken, template).problems():
    print("-", problem)

## 6. The outputs folder

A project has two kinds of documents. `sources/` is the knowledge base: it is
indexed and searched. `outputs/` holds what QAChat writes, such as research
briefings, and anything you add there to work on. It is **never indexed**,
so a generated report can never come back as a source. In the desktop app,
right-click `outputs` or `templates` in the workspace to add, replace, or
delete files.

This tutorial uses a small project folder with one paper in `outputs/`. The
tool can also summarize any paper among an indexed project's sources.

In [ ]:
from qachat.tools.outputs import output_folder

project_root = root / "sandbox" / "tutorial_160_custom_tool"
outputs = output_folder(project_root)
print(outputs)

Choose the included example paper.

In [ ]:
paper = root / "example_repos" / "literature_repo" / "papers"
paper = paper / "patil25deep.pdf"
print(paper)

Copy it into the tutorial project's outputs folder.

In [ ]:
import shutil

shutil.copy2(paper, outputs / paper.name)

Save the custom template in the tutorial project's templates folder.

In [ ]:
from qachat.tools.report_template import templates_folder

templates = templates_folder(project_root)
(templates / "lab_summary.md").write_text(lab_summary, encoding="utf-8")

Load the copied paper as passages, without building a retrieval index.

In [ ]:
from qachat.tools.outputs import load_output

document = load_output(project_root, paper.name)
print(f"{document.path}: {len(document.passages)} passages, "
      f"about {document.estimated_tokens:,} tokens")

Inspect the available templates.

In [ ]:
print(sorted(p.name for p in templates.glob("*.md")))

## 7. Nothing bibliographic is invented

Small models often invent plausible references. This tool never lets them
write one. The citation comes from the PDF's metadata and first page, and
the paper's reference list is split into entries by code:

In [ ]:
from qachat.tools.generate_paper_summary import paper_citation

paper_file = project_root / document.path
print(paper_citation(paper_file, document.passages))

Extract the reference entries separately and inspect a sample.

In [ ]:
from qachat.tools.generate_paper_summary import reference_entries

entries = reference_entries(document.passages)
print(f"{len(entries)} reference entries, for example:")
for entry in entries[:3]:
    print("-", entry)

For a `{references}` section the model is shown the numbered entries and
replies with numbers only. Code copies those entries word for word;
numbers outside the list are ignored.

## 8. Run the tool with stand-in services

Before using a real model, run the tool with scripted replies. This is how
QAChat's tests work. The tool writes the headline and **each section in
its own call**, and Python assembles the headings, so a section can never
go missing or out of order. Each piece is checked as it arrives.

The scripted model below answers the reference question with "3, 99": entry
3 is copied, and 99, which does not exist, is ignored. The template asks for
two items, so the entry closest to the reading notes fills the second place.

In [ ]:
def scripted_model(system, prompt, max_new_tokens):
    if "take notes" in system:
        return "- The method is called DIVA [1]"
    if "choose references" in system:
        return "3, 99"
    return "Summarized from the notes [1]."

Create the service context. Each field supplies one capability.
The scripted model replaces real calls, and `steps` collects progress.

In [ ]:
from qachat.trace_logging import NullTraceLogger

steps = []

def make_context(model):
    return ToolContext(
        project_root=project_root,
        chunks=[],
        graph=qachat.CodeGraph(entities=(), relations=()),
        rank=lambda text: [],
        retrieval_count=20,
        unsearchable=lambda indexed: [],
        source_files=lambda: {},
        ask_router=model,
        ask_model=model,
        write=model,
        model_id="scripted",
        progress=lambda message, **details: steps.append(message),
        done=lambda message, **details: steps.append(message),
        confirm=None,
        trace=NullTraceLogger(),
        trace_turn_id=None,
    )

In [ ]:
from qachat.tools import ToolUse, run_tool
from qachat.tools.generate_paper_summary import SummarySettings

use = ToolUse("generate_paper_summary", SummarySettings())

Name the paper explicitly as well as the template. After a rerun,
`outputs/` also contains earlier reports, so "my paper" is ambiguous.

In [ ]:
question = f"turn {paper.name} in outputs into a lab summary"
request = ToolRequest(question, question)

Run the tool with scripted services. It saves a tutorial report
to `outputs/` without calling a real model.

In [ ]:
result = run_tool(use, request, make_context(scripted_model))

Inspect the steps reported by the tool.

In [ ]:
for step in steps:
    print("·", step)

Inspect the chosen template and saved location.

In [ ]:
if "saved" not in result.summary:
    raise RuntimeError(
        "The tool did not save a report. Its explanation: " + result.answer
    )
print("template:", result.summary["template"])
print("saved:", result.summary["saved"])

Read the generated report.

In [ ]:
saved = project_root / result.summary["saved"]
print(saved.read_text(encoding="utf-8"))

## 9. Optional: run it with a real model

The scripted example above is enough to learn the tool workflow.
By default, this section does not load or download any model. Leave
`RUN_REAL_MODEL = False` for a quick run of the whole notebook.

To try real generation later, change the flag below to `True` and run
this section. The first run may download the default chat model, and
generating a research briefing may take several minutes. The tool reads
the paper in parts, writes from the notes, checks each piece, and saves
the briefing.

In [ ]:
RUN_REAL_MODEL = False
real_result = None
chat = None
if not RUN_REAL_MODEL:
    print("Skipping optional real-model testing.")

In [ ]:
backend = qachat.choose_backend(qachat.BACKEND_AUTO)
print("Backend:", backend)

Select the default model for that backend.

In [ ]:
model_id = (
    qachat.DEFAULT_CHAT_MODEL_MLX
    if backend == qachat.BACKEND_MLX
    else qachat.DEFAULT_CHAT_MODEL_TORCH
)
print("Model:", model_id)

Load the model; the first run may download it.

In [ ]:
if RUN_REAL_MODEL:
    chat = qachat.ChatSession(
        model_id,
        backend=backend,
        thinking=False,
        temperature=0.0,
        allow_download=True,
    )

Adapt the model to the tool's callable interface. `side_reply` restores
the conversation after each call.

In [ ]:
from qachat.answering import side_reply

def model(system, prompt, max_new_tokens):
    return side_reply(chat, system, prompt, max_new_tokens=max_new_tokens)

In [ ]:
briefing = ToolUse(
    "generate_paper_summary", SummarySettings(repair_attempts=2)
)

Name the same paper explicitly and request the shipped
research-briefing template.

In [ ]:
question = f"turn {paper.name} in outputs into a research briefing"
request = ToolRequest(question, question)

Run the same tool with real model services. This may take several minutes.

In [ ]:
if RUN_REAL_MODEL:
    steps.clear()
    real_result = run_tool(briefing, request, make_context(model))

Inspect the progress after the run.

In [ ]:
if RUN_REAL_MODEL:
    print("\n".join(f"· {step}" for step in steps))

In [ ]:
if real_result is not None:
    if "saved" not in real_result.summary:
        raise RuntimeError(
            "The tool did not save a report. Its explanation: "
            + real_result.answer
        )
    print("valid:", real_result.summary["valid"])
    print("problems:", real_result.summary["problems"])

Display the saved report.

In [ ]:
if real_result is not None:
    saved = project_root / real_result.summary["saved"]
    display(Markdown(saved.read_text(encoding="utf-8")))

Release the model before the model-free custom tool example.

In [ ]:
if chat is not None:
    chat.close()

In the desktop app, the same tool runs when you ask a literature project
something like *"Turn the DIVA paper into a research briefing"*: the router
picks the `paper_summary` intent, and the answer says where the Markdown and
a one-page PDF were saved, then shows the briefing.

## 10. Write your own tool

A new tool needs one module, one registry entry, and an intent that names
it. Below is a complete, minimal tool that needs no model: it counts the
words of a file in `outputs/`. Copy this shape for your own tools.

Define the settings: how many frequent words to include in the result.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True, slots=True)
class WordCountSettings:
    """Settings of the example word-count tool."""

    top_words: int = 5

Validate unknown fields and the setting's allowed range.

In [ ]:
from qachat.tools.contract import check_int, reject_unknown

def parse_settings(data: dict, where: str) -> WordCountSettings:
    """Validate settings from a project type."""

    reject_unknown(where, data, ("top_words",))
    return WordCountSettings(
        top_words=check_int(where, data, "top_words", 5, 1, 50)
    )

Define the operation. The request supplies a file name in `outputs/`,
so it does not depend on a notebook-only global variable. This example
counts whitespace-separated words, not linguistic tokens.

In [ ]:
from collections import Counter

def run(settings, request, context) -> ToolResult:
    """Count words and list the most frequent ones."""

    document = load_output(context.project_root, request.text.strip())
    words = " ".join(p.text for p in document.passages).split()
    frequent = Counter(word.casefold() for word in words)
    top = frequent.most_common(settings.top_words)
    context.done(f"Counted {len(words)} words")
    answer = f"{document.path} has about {len(words):,} words."
    answer += "\nFrequent words: " + ", ".join(
        f"{word} ({count})" for word, count in top
    )
    return ToolResult(answer, (), {"name": "word_count"})

In [ ]:
settings = parse_settings({"top_words": 3}, "example.intents.count.tool")
print(settings)

Give this example tool the file name as its request text.

In [ ]:
request = ToolRequest(paper.name, paper.name)

Run the operation and inspect its answer. No model is needed.

In [ ]:
print(run(settings, request, make_context(scripted_model)).answer)

To make it available in QAChat:

1. Save the module as `qachat/tools/word_count.py`.
2. Register it in `qachat/tools/registry.py`: add `"word_count"` to
   `ToolName`, a `TOOL_WORD_COUNT` constant, and a `Tool(...)` entry with its
   `parse_settings` and `run`.
3. Name it in an intent of `qachat/project_types/literature.json`
   (tutorial 150 explains the other intent fields):

```json
"word_count": {
  "description": "Count the words of a paper in the outputs folder.",
  "history": {"policy": "none"},
  "retrieval": {"mode": "none"},
  "tool": {"name": "word_count", "top_words": 3}
}
```

Registration makes a tool *available*; the project type decides *when* it is
used. Ideas that fit the same pattern: comparing two papers, extracting
methods or statistics, screening papers against criteria, or checking a
manuscript against a reporting guideline.